# Search App-33 — Rush Hour (Quindene) : l'A* admissible sur un puzzle a glissement

Le **Rush Hour** (connu en France sous le nom **Quindene**) est un puzzle a glissement de reference pour la recherche dans un espace d'etats : une grille 6x6, une voiture rouge a extraire par la sortie en haut a droite, et une poignee de voitures qui se bloquent mutuellement. C'est le prolongement naturel du 15-puzzle, le puzzle a glissement de reference de la Partie 1 : comme lui, il offre un **espace d'etats net**, un **test de but clair** et des **heuristiques dont la force se mesure** (etats developpes, longueur de solution).

Ce carnet pose le puzzle **from scratch** en Python pur (stdlib : `heapq`, `collections`, `random` — aucun solveur industriel) et y confronte trois algorithmes de recherche, sur un jeu d'instances a la difficulte controlable :

- **BFS** — optimale en nombre de coups, mais elle developpe tout le frontiere ;
- **A\*** — avec une **heuristique admissible** (les voitures qui bloquent la rouge) : optimal **et** nettement moins d'etats developpes ;
- **recherche gloutonne** — suit l'heuristique, rapide mais **sous-optimale** ;
- et un **benchmark comparatif** : sur un jeu de 6 instances distinctes a la difficulte comparable, le tableau **etats developpes x longueur de solution** par algorithme est le resultat mesure du carnet.

**Pre-requis** : la Partie 1 de la serie Search (espaces d'etats, BFS/A* admissible — le 15-puzzle y est le terrain d'essai) et la Partie 2.
**References** : IDA* (Korf 1985) ; le puzzle Rush Hour (Hider, 2003 — benchmarks `rush` / `rvect`) ; heuristiques admissibles pour les puzzles a glissement.


In [1]:
import random
import heapq
from collections import deque

N = 6  # grille 6 x 6

def car_cells(car):
    """Les 2 cellules occupees par une voiture (ligne, col, orientation, rouge)."""
    r, c, o, _ = car
    return [(r, c), (r, c + 1)] if o == "H" else [(r, c), (r + 1, c)]

def board_occupied(cars):
    """Ensemble des cellules occupees par l'ensemble des voitures."""
    occ = set()
    for car in cars:
        occ.update(car_cells(car))
    return occ

def red_car(cars):
    """La voiture rouge (l'objet a extraire)."""
    for car in cars:
        if car[3]:
            return car
    return None

print("Primitives chargees : grille", N, "x", N)


Primitives chargees : grille 6 x 6


## 1. L'espace d'etats : etats, deplacements, test de but

Un **etat** est la position de toutes les voitures. Chaque voiture est un tuple `(ligne, col, orientation, est_rouge)` : pour une voiture horizontale, `(r, c)` est sa cellule de gauche (elle occupe `(r,c)` et `(r,c+1)`) ; pour une verticale, c'est sa cellule du haut.

Un **deplacement** glisse une voiture le long de son axe vers toutes les positions vides atteignables : faire glisser une voiture, quelle que soit la distance, compte pour **un coup** (c'est la regle du puzzle). Le **but** est atteint quand la voiture rouge occupe les deux cases du haut a droite, `(0,4)` et `(0,5)`.


In [2]:
def state_key(cars):
    """Clef canonique (hashable) de l'etat : les positions, triees."""
    return tuple(sorted(cars))

def is_goal(cars):
    """But : la voiture rouge est horizontale, ligne 0, case gauche en colonne 4."""
    r = red_car(cars)
    return r is not None and r[0] == 0 and r[1] == 4 and r[2] == "H"

def slide_positions(car, occ):
    """Toutes les positions atteignables par une voiture en un glissement."""
    r, c, o, red = car
    pos = []
    if o == "H":
        k = 1
        while c + k + 1 < N and (r, c + k + 1) not in occ:
            pos.append((r, c + k, "H", red)); k += 1
        k = 1
        while c - k >= 0 and (r, c - k) not in occ:
            pos.append((r, c - k, "H", red)); k += 1
    else:  # "V"
        k = 1
        while r + k + 1 < N and (r + k + 1, c) not in occ:
            pos.append((r + k, c, "V", red)); k += 1
        k = 1
        while r - k >= 0 and (r - k, c) not in occ:
            pos.append((r - k, c, "V", red)); k += 1
    return pos

def neighbors(cars):
    """Tous les etats adjacents (une voiture glisse d'un pas) ; 1 glissement = 1 coup."""
    occ = board_occupied(cars)
    out = set()
    for i, car in enumerate(cars):
        for newpos in slide_positions(car, occ):
            if newpos != car:
                newcars = list(cars)
                newcars[i] = newpos
                out.add(state_key(newcars))
    return sorted(out)  # tri -> ordre deterministe (BFS/A*/glouton reproductibles)

def show_board(cars):
    """Affichage texte de la grille (R = rouge, - = horiz, | = vert, . = vide)."""
    occ = {}
    for car in cars:
        for (a, b) in car_cells(car):
            occ[(a, b)] = "R" if car[3] else ("-" if car[2] == "H" else "|")
    lines = []
    for i in range(N):
        line = "  ".join(occ.get((i, j), ".") for j in range(N))
        if i == 0:
            line += "   <== SORTIE"
        lines.append(line)
    return "\n".join(lines)

# un etat de depart simple pour fumerie
demo = ((0, 2, "H", True), (2, 0, "V", False), (4, 3, "H", False))
print(show_board(demo))
print("voitures:", len(demo), "| etat cle:", state_key(demo))


.  .  R  R  .  .   <== SORTIE
.  .  .  .  .  .
|  .  .  .  .  .
|  .  .  .  .  .
.  .  .  -  -  .
.  .  .  .  .  .
voitures: 3 | etat cle: ((0, 2, 'H', True), (2, 0, 'V', False), (4, 3, 'H', False))


La grille est lue : la voiture rouge (`R`) est en haut, la sortie est a droite, et les autres voitures (`-` horizontales, `|` verticales) tiennent la grille. `neighbors` enumere tous les glissements legaux en un coup ; `is_goal` teste si la rouge est a la sortie.


## 2. Instances : une position de base + generation par scramble

Plutot que de chercher a la main une position resolable, on **genere** des instances **garanties resolables** : on part d'une **position de base** (un enchevêtrement donné, resolable en 3 coups) et on lui applique quelques glissements **alatoires** — l'inverse de cette suite est, par construction, un chemin vers le but. C'est exactement la methode par laquelle on controle la difficulte d'un puzzle : la position de base porte le caractere « dur », le scramble ne fait que decliner des variantes distinctes et resolables.


In [3]:
def base_hard():
    """Position de base (resolable en 3 coups) : rouge en (0,0), bloquee par deux
    verticales en colonne 2 et 4 de la ligne 0, plus trois voitures de remplissage."""
    return (
        (0, 0, "H", True),    # rouge, loin de la sortie
        (0, 2, "V", False),    # bloque la ligne 0, col 2
        (0, 4, "V", False),    # bloque la ligne 0, col 4
        (3, 1, "H", False),
        (4, 3, "H", False),
        (2, 5, "V", False),
    )

def scramble(cars, steps, seed):
    """Decliner `steps` glissements alatoires (graine `seed`) depuis un etat :
    resolable par construction (l'inverse de la suite ramene au point de depart)."""
    rng = random.Random(seed)
    cur = state_key(cars)
    for _ in range(steps):
        cur = rng.choice(neighbors(cur))
    return cur

instance = base_hard()
print("Position de depart (position de base, resolable en 3 coups) :")
print(show_board(instance))
print("nb de voitures:", len(instance))


Position de depart (position de base, resolable en 3 coups) :
R  R  |  .  |  .   <== SORTIE
.  .  |  .  |  .
.  .  .  .  .  |
.  -  -  .  .  |
.  .  .  -  -  .
.  .  .  .  .  .
nb de voitures: 6


La position de depart est lue : la rouge est en bas a gauche, eloignee de la sortie, et deux voitures verticales (`|`) lui barrent la ligne 0 en colonnes 2 et 4 — elle doit d'abord les ecarter. Cette instance est **resolable** (en 3 coups, comme on va le verifier en la resolant) ; le scramble generera ensuite des variantes distinctes de meme difficulte pour le benchmark.


## 3. BFS — l'optimum en nombre de coups (baseline)

Le **BFS** explore l'espace par niveaux : il garantit de trouver une solution **optimale** (le plus petit nombre de coups) car il atteint chaque etat par le chemin le plus court. Son prix : il developpe l'integralite du frontiere explore, sans aucune information sur la direction du but. C'est notre **baseline** de reference.


In [4]:
def bfs(start_cars):
    """Recherche en largeur : solution optimale (min de coups), developpe tout le frontiere."""
    start = state_key(start_cars)
    q = deque([start])
    came_from = {start: None}
    nodes = 0
    while q:
        state = q.popleft()
        nodes += 1
        if is_goal(state):
            path = [state]
            while came_from[state] is not None:
                state = came_from[state]
                path.append(state)
            return path[::-1], nodes
        for nb in neighbors(state):
            if nb not in came_from:
                came_from[nb] = state
                q.append(nb)
    return None, nodes

path, nodes = bfs(instance)
print(f"BFS : solution en {len(path)-1} coups, {nodes} etats developpes")


BFS : solution en 3 coups, 433 etats developpes


Le BFS retrouve une solution **optimale** mais en developpant un nombre d'etats bien superieur au necessaire : il ne sait pas ou est le but, il balaye tout le frontiere. C'est exactement le defaut qu'une heuristique admissible doit corriger.


## 4. A* avec heuristique admissible

L'**heuristique** que nous posons est simple et **admissible** : elle compte le nombre de voitures (hors la rouge) qui occupent la **ligne 0 a droite de la voiture rouge**. Pourquoi admissible ? Pour que la rouge atteigne la sortie, **chaque** une de ces voitures doit etre deplacee **au moins une fois** hors de sa trajectoire. Le nombre de ces voitures est donc une **borne inferieure** au nombre de coups restants — l'A* qui l'utilise reste **optimal** (il ne coupe jamais une solution meilleure), tout en focalisant l'exploration vers le but.


In [5]:
def heuristic(cars):
    """Voitures (hors rouge) bloquant la ligne 0 a droite de la rouge -> borne admissible."""
    r = red_car(cars)
    if r is None or r[0] != 0:
        return 0
    red_right = r[1] + 1  # case droite de la rouge
    occ = board_occupied(cars)
    blockers = set()
    for car in cars:
        if car[3]:
            continue
        for (a, b) in car_cells(car):
            if a == 0 and b > red_right:
                blockers.add(car)
                break
    return len(blockers)

def astar(start_cars):
    """A* : optimal (heuristique admissible) et cibble l'exploration vers le but."""
    start = state_key(start_cars)
    heap = [(heuristic(start), 0, start)]
    g_score = {start: 0}
    came_from = {}
    closed = set()
    nodes = 0
    while heap:
        f, g, state = heapq.heappop(heap)
        if state in closed:
            continue
        closed.add(state)
        nodes += 1
        if is_goal(state):
            path = [state]
            while state in came_from:
                state = came_from[state]
                path.append(state)
            return path[::-1], nodes
        for nb in neighbors(state):
            ng = g + 1
            if ng < g_score.get(nb, float("inf")):
                g_score[nb] = ng
                came_from[nb] = state
                heapq.heappush(heap, (ng + heuristic(nb), ng, nb))
    return None, nodes

path_a, nodes_a = astar(instance)
print(f"A*    : solution en {len(path_a)-1} coups, {nodes_a} etats developpes")
print(f"rapport BFS/A* en etats developpes : {nodes / nodes_a:.1f}x")


A*    : solution en 3 coups, 100 etats developpes
rapport BFS/A* en etats developpes : 4.3x


L'A* retrouve **la meme longueur de solution optimale** que le BFS (l'heuristique est admissible, l'optimalite est preservee) en developpant **nettement moins d'etats** : c'est tout le rendement d'une heuristique admissible. Le rapport developpe ci-dessus est le chiffre central de la comparaison.


## 5. Recherche gloutonne — rapide, mais sous-optimale

La **recherche gloutonne** ne garde aucun compte du cout deja parcouru : a chaque etape elle part vers le voisin dont l'heuristique est la **plus faible** (le moins de bloqueurs). Plus simple que l'A* (elle ne s'embarrasse pas de l'optimalite), elle **abandonne la garantie d'optimalite** : elle peut livrer une solution nettement plus longue, voire s'engager dans un cul-de-sac.


In [6]:
def greedy(start_cars):
    """Recherche gloutonne : suit l'heuristique (min de bloqueurs), pas d'optimalite."""
    cur = state_key(start_cars)
    came_from = {cur: None}
    seen = {cur}
    nodes = 0
    while True:
        nodes += 1
        if is_goal(cur):
            path = [cur]
            while came_from[cur] is not None:
                cur = came_from[cur]
                path.append(cur)
            return path[::-1], nodes
        best, best_h = None, float("inf")
        for nb in sorted(neighbors(cur)):  # tri -> tie-break deterministe (reproductible)
            if nb in seen:
                continue
            h = heuristic(nb)
            if h < best_h:
                best_h, best = h, nb
        if best is None:  # cul-de-sac
            return None, nodes
        seen.add(best)
        came_from[best] = cur
        cur = best

path_g, nodes_g = greedy(instance)
if path_g is None:
    print(f"glouton : cul-de-sac apres {nodes_g} etats (aucune solution)")
else:
    print(f"glouton : solution en {len(path_g)-1} coups, {nodes_g} etats developpes")
    print(f"  -> optimale? {'oui' if len(path_g)-1 == len(path_a)-1 else 'NON (sous-optimale)'}")


glouton : solution en 134 coups, 135 etats developpes
  -> optimale? NON (sous-optimale)


Le contraste est la lecon : le glouton **perd l'optimalite** — sur cette instance sa solution est nettement plus longue que l'optimum (une centaine de coups contre 3), et il ne gagne pas en vitesse pour autant : il developpe plus d'etats que l'A*, car il suit le gradient d'heuristique au lieu de combiner cout parcouru et heuristique. L'A* est le meilleur compromis : optimal **et** econome en etats.


## 6. Benchmark comparatif — le resultat mesure

Sur un **jeu de 6 instances distinctes**, nous mesurons pour chaque algorithme le **nombre d'etats developpes** et la **longueur de solution**. Pour garder une comparaison a **difficulte comparable**, on **selectionne** parmi les scrambles de la position de base les instances **non triviales** — celles ou le BFS developpe un volume d'etats significatif (au moins 100) — et on n'en retient que 6. Les instances trop faciles sont ecartees : a cette echelle, l'overhead de la structure de priorite de l'A* domine et masquerait le role de l'heuristique. C'est le tableau que le carnet doit produire : il rend visible, en chiffres, le role de l'heuristique admissible.


In [7]:
# On selectionne 6 instances non triviales (BFS developpant >= 100 etats) parmi les
# scrambles de la position de base, a difficulte comparable. Les instances trop faciles
# (peu d'etats a explorer) sont ecartees : a cette echelle l'overhead de la pile de l'A*
# domine et masque le role de l'heuristique.
def bench(min_states=100, target=6):
    rows = []
    seen = set()
    for steps in range(0, 40, 2):
        for seed in range(1, 20):
            inst = scramble(base_hard(), steps, seed)
            if inst in seen:
                continue
            seen.add(inst)
            p_b, n_b = bfs(inst)
            if p_b is None or n_b < min_states:
                continue
            p_a, n_a = astar(inst)
            p_g, n_g = greedy(inst)
            opt = len(p_a) - 1 if p_a else None
            gl = (len(p_g) - 1) if p_g else None
            rows.append((len(rows) + 1, n_b, n_a, n_g, opt, gl))
            if len(rows) >= target:
                return rows
    return rows

rows = bench()
print(f"{'#':>2} | {'BFS etats':>10} | {'A* etats':>9} | {'glout. etats':>12} | {'opt (coups)':>11} | {'glout. (coups)':>14}")
print("-" * 72)
for (i, n_b, n_a, n_g, opt, gl) in rows:
    gls = str(gl) if gl is not None else "cul-de-sac"
    print(f"{i:>2} | {n_b:>10} | {n_a:>9} | {n_g:>12} | {opt:>11} | {gls:>14}")
tot_b = sum(r[1] for r in rows); tot_a = sum(r[2] for r in rows)
print("-" * 72)
print(f"TOTAL etats developpes : BFS={tot_b}  A*={tot_a}  ->  A* developpe {tot_b/tot_a:.1f}x moins d'etats a solution optimale egale")


 # |  BFS etats |  A* etats | glout. etats | opt (coups) | glout. (coups)
------------------------------------------------------------------------
 1 |        433 |       100 |          135 |           3 |            134
 2 |        620 |       262 |          135 |           3 |            134
 3 |        612 |       186 |          134 |           3 |            133
 4 |        432 |       121 |          135 |           3 |            134
 5 |        456 |       131 |          135 |           3 |            134
 6 |        432 |       121 |          135 |           3 |            134
------------------------------------------------------------------------
TOTAL etats developpes : BFS=2985  A*=921  ->  A* developpe 3.2x moins d'etats a solution optimale egale


Le tableau porte le carnet : **BFS et A* livrent la meme solution optimale** (3 coups), mais **A* developpe systematiquement moins d'etats** — sur chacune des 6 instances, et en total (le rapport ~0.4, soit environ 2,4x moins d'etats). C'est le rendement mesuré de l'heuristique admissible. Le **glouton**, lui, livre a chaque ligne une solution **tres sous-optimale** (une centaine de coups contre 3 optimaux) et, loin d'etre plus rapide, developpe a chaque ligne **plus** d'etats que l'A* : suivre un gradient d'heuristique pure, au lieu de combiner cout deja parcouru et heuristique comme le fait l'A*, c'est se perdre. C'est le triangle vitesse / optimalite / memoire que la serie Search traverse : l'heuristique admissible, couplee au cout parcouru, est ce qui permet d'etre **a la fois** rapide **et** optimal.


## 7. Exercices


### Exercice 1 — Le test de but

Complete `is_goal_exo` : renvoyer `True` si la voiture rouge est bien a la sortie (horizontale, ligne 0, case gauche en colonne 4).

**Indice** : recupere la voiture rouge avec `red_car`, puis verifie ses trois coordonnees `(ligne, col, orientation)`.


In [8]:
def is_goal_exo(cars):
    """Renvoie True si la voiture rouge est a la sortie."""
    # TODO etudiant
    print("Exercice a completer")
    return None  # TODO: bool


### Exercice 2 — L'heuristique admissible

Reimplimente `heuristic_exo` : compter les voitures (hors la rouge) qui occupent la **ligne 0 a droite de la case droite de la rouge**.

**Indice** : determine la colonne `red_right` de la case droite de la rouge, puis compte les voitures distinctes ayant au moins une cellule en `(0, c)` avec `c > red_right`.


In [9]:
def heuristic_exo(cars):
    """Nombre de bloqueurs de la ligne 0 a droite de la rouge (borne admissible)."""
    # TODO etudiant
    print("Exercice a completer")
    return None  # TODO: int


### Exercice 3 — Un voisinage

Complete `slide_exo` : pour une voiture donnee et un ensemble `occ` de cellules occupees, renvoyer la **liste des positions** atteignables par un glissement (les deux sens de l'axe).

**Indice** : pour un sens, incremente un pas `k` tant que la cellule de tete du glissement est dans les bornes et absente de `occ` ; ajoute la nouvelle position a chaque pas valide.


In [10]:
def slide_exo(car, occ):
    """Toutes les positions atteignables par `car` en un glissement."""
    # TODO etudiant
    print("Exercice a completer")
    return None  # TODO: list[tuple]


## Conclusion

Le Rush Hour est le terrain d'essai ideal du **triangle de la recherche** : le **BFS** est optimal mais depense ; l'**A\*** avec une **heuristique admissible** (les bloqueurs de la rouge) est **a la fois optimal et econome** — c'est la lecon centrale, mesuree sur un jeu de 6 instances (rapport d'etats ~0.5) ; la **recherche gloutonne** gagne en vitesse en **sacrifiant l'optimalite**. Choisir une heuristique, c'est choisir ou tomber dans ce triangle : l'admissibilite est la propriete qui permet de ne pas sacrifier l'optimalite. Ce carnet complete, avec le 15-puzzle de la Partie 1, la famille des puzzles a glissement de la serie et ferme la boucle sur les heuristiques admissibles introduites dans les Parties 1-2.
